# Retail E-commerce Sales Analytics Pipeline
## Phase 3: Silver Stage 2 — SCD Type 2 Dimensions

This notebook builds `dim_customer_scd2` and `dim_product_scd2` — dimension
tables that keep the **full history** of every change, using SCD Type 2:
- `surrogate_key` — a unique key per historical version of a record
- `effective_start_date` / `effective_end_date` — the date range this version was active
- `is_current` — whether this is the currently active version
- `hash_value` — a fingerprint of the tracked attributes, used to detect changes

**Approach:** We do an initial load from the Silver Stage 1 clean tables, then
process each day's CDC file **in chronological order**, applying the classic
2-step MERGE pattern: (1) expire the old active row if attributes changed,
(2) insert a new active row with the updated attributes.

In [0]:
CATALOG = "retail_demo"    
RAW_SCHEMA = "raw"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SILVER_SCHEMA}")

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import warnings
warnings.filterwarnings("ignore", message=".*No Partition Defined for Window operation.*")

FAR_FUTURE_DATE = "9999-12-31"
FAR_PAST_DATE = "1900-01-01"

print("Ready.")

Ready.


## Helper Functions
These are generic so we can reuse the exact same SCD2 logic for both
customers and products, instead of writing it twice.

In [0]:
def initial_scd2_load(clean_df, business_key, attr_cols, table_name):
    """
    Build the very first version of an SCD2 dimension table from a Silver
    Stage 1 clean DataFrame. Every row becomes version 1, active from the
    far past to the far future.

    The dimension table is kept "narrow" on purpose: only the business key,
    the tracked attributes, and the SCD2 bookkeeping columns. We deliberately
    do NOT carry over Bronze/Silver audit columns or untracked columns like
    signup_date/created_date — this keeps the schema identical to what the
    CDC files produce, which is what makes the day-by-day MERGE logic below
    work without column mismatches.
    """
    hash_expr = F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in attr_cols]), 256)

    initial_df = (
        clean_df
        .select(business_key, *attr_cols)
        .withColumn("hash_value", hash_expr)
        .withColumn("effective_start_date", F.to_date(F.lit(FAR_PAST_DATE)))
        .withColumn("effective_end_date", F.to_date(F.lit(FAR_FUTURE_DATE)))
        .withColumn("is_current", F.lit(True))
        .withColumn("surrogate_key", F.row_number().over(Window.orderBy(business_key)))
    )

    (initial_df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name))

    print(f"Initial load into {table_name}: {spark.table(table_name).count()} rows")


def get_next_surrogate_key_offset(table_name):
    """Get the current max surrogate_key so new keys never collide."""
    max_key = spark.table(table_name).agg(F.max("surrogate_key")).collect()[0][0]
    return max_key if max_key is not None else 0


def apply_scd2_merge(cdc_day_df, table_name, business_key, attr_cols, effective_date_col="effective_date"):
    """
    Apply one day's worth of CDC changes to an SCD2 dimension table using the
    2-step MERGE pattern:
      Step 1: expire the currently active row for any business key whose
              attribute hash has changed.
      Step 2: insert a brand-new active row (new surrogate_key) for every
              business key that is new OR whose attributes changed.
    """
    hash_expr = F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in attr_cols]), 256)

    # Clean & dedupe this day's CDC batch: keep the latest ingestion per business key
    day_window = Window.partitionBy(business_key).orderBy(F.col("ingestion_ts").desc())
    cdc_clean = (
        cdc_day_df
        .withColumn("effective_date_parsed", F.expr(f"try_to_date({effective_date_col}, 'yyyy-MM-dd')"))
        .filter(F.col("effective_date_parsed").isNotNull())
        .filter(F.col(business_key).isNotNull())
        .withColumn("incoming_hash", hash_expr)
        .withColumn("row_num", F.row_number().over(day_window))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    n_incoming = cdc_clean.count()
    if n_incoming == 0:
        print(f"  No valid CDC rows for {table_name} on this day — skipping.")
        return

    dim_dt = DeltaTable.forName(spark, table_name)

    # --- Step 1: expire active rows whose hash has changed ---
    (dim_dt.alias("target")
        .merge(
            cdc_clean.alias("source"),
            f"target.{business_key} = source.{business_key} AND target.is_current = true"
        )
        .whenMatchedUpdate(
            condition="target.hash_value <> source.incoming_hash",
            set={
                "effective_end_date": "date_sub(source.effective_date_parsed, 1)",
                "is_current": "false",
            }
        )
        .execute())

    # --- Step 2: insert new active rows for changed OR brand-new business keys ---
    current_active = (spark.table(table_name)
                       .filter(F.col("is_current") == True)
                       .select(business_key, "hash_value"))

    to_insert = (
        cdc_clean.alias("c")
        .join(current_active.alias("a"), on=business_key, how="left")
        .filter(F.col("a.hash_value").isNull() | (F.col("a.hash_value") != F.col("c.incoming_hash")))
    )

    n_new = to_insert.count()
    if n_new == 0:
        print(f"  No new/changed rows to insert for {table_name}.")
        return

    key_offset = get_next_surrogate_key_offset(table_name)

    new_rows = (
        to_insert
        .withColumn("hash_value", F.col("incoming_hash"))
        .withColumn("effective_start_date", F.col("effective_date_parsed"))
        .withColumn("effective_end_date", F.to_date(F.lit(FAR_FUTURE_DATE)))
        .withColumn("is_current", F.lit(True))
        .withColumn("surrogate_key", key_offset + F.row_number().over(Window.orderBy(business_key)))
        .select(business_key, *attr_cols, "hash_value", "effective_start_date",
                 "effective_end_date", "is_current", "surrogate_key")
    )

    (new_rows.write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(table_name))

    print(f"  Expired changed rows + inserted {n_new} new active row(s) into {table_name}")

## 1. Customer Dimension — Initial Load

In [0]:
spark.sql(f"SHOW TABLES IN {CATALOG}.{SILVER_SCHEMA}").show(truncate=False)

+--------+-----------------------+-----------+
|database|tableName              |isTemporary|
+--------+-----------------------+-----------+
|silver  |dim_customer_scd2      |false      |
|silver  |dim_product_scd2       |false      |
|silver  |dim_store              |false      |
|silver  |quarantine_customers   |false      |
|silver  |quarantine_orders      |false      |
|silver  |quarantine_products    |false      |
|silver  |silver1_customers_clean|false      |
|silver  |silver1_orders_clean   |false      |
|silver  |silver1_products_clean |false      |
+--------+-----------------------+-----------+



In [0]:
silver1_customers = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver1_customers_clean")

customer_attr_cols = ["customer_name", "city", "segment", "gender", "status"]

initial_scd2_load(
    clean_df=silver1_customers,
    business_key="customer_id",
    attr_cols=customer_attr_cols,
    table_name=f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
)

Initial load into retail_demo.silver.dim_customer_scd2: 2500 rows


In [0]:
bronze_customers_cdc = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_cdc")

In [0]:
display(spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2").limit(5))

customer_id,customer_name,city,segment,gender,status,hash_value,effective_start_date,effective_end_date,is_current,surrogate_key
C00001,Customer_1,Delhi,Regular,Other,active,590d9ddd4a8541ad8b91a8cb8f498da0fc9839b20d3da10240c6f5ee4490cf2f,1900-01-01,9999-12-31,true,1
C00002,Customer_2,Bengaluru,Silver,Other,active,5283080b9504f2419a71f5a1f247bdb486358edd5c7a2fa6e5e92a49af9389b4,1900-01-01,9999-12-31,true,2
C00003,Customer_3,Gurugram,Platinum,M,active,6632bb74654c4055611912a858a0cf14ca579e3fd1f90cf43aeeeeb048a9be53,1900-01-01,9999-12-31,true,3
C00004,Customer_4,Bengaluru,Silver,Other,active,9baf25bb13d3c35cf1c8d8d7f957c9806ee24a312c1db0a18f65b482514dd244,1900-01-01,9999-12-31,true,4
C00005,Customer_5,Ahmedabad,Silver,Other,inactive,151703a73ed3fb2f2dbc7b529bf5a981ae8551d89dd8b35b9e32e0dab1f7aaa9,1900-01-01,9999-12-31,true,5


In [0]:
bronze_customers_cdc.printSchema()
display(bronze_customers_cdc.limit(5))

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)



customer_id,customer_name,city,segment,gender,signup_date,status,effective_date,operation,_rescued_data,source_file,ingestion_ts,load_type
C00606,Customer_606,Kolkata,Silver,Other,2025-10-09,active,not_a_date,UPDATE,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,2026-07-11T15:18:02.705Z,incremental
C00527,Customer_527,Bengaluru,Regular,M,2024-04-10,inactive,2026-04-25,UPDATE,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,2026-07-11T15:18:02.705Z,incremental
C00660,Customer_660,Jaipur,Regular,F,2024-10-16,active,2026-04-25,UPDATE,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,2026-07-11T15:18:02.705Z,incremental
C01018,Customer_1018,Jaipur,Platinum,F,2025-04-06,active,2026-04-25,UPDATE,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,2026-07-11T15:18:02.705Z,incremental
C00781,Customer_781,Pune,Regular,F,2025-12-01,active,2026-04-25,UPDATE,null,/Volumes/workspace/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,2026-07-11T15:18:02.705Z,incremental


## 2. Customer Dimension — Apply CDC Day by Day
We process the 3 incremental days **in chronological order** — this matters
because each day's changes should be applied on top of the previous day's
resulting state.

In [0]:
bronze_customers_cdc = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_cdc")

cdc_days = sorted([
    row.effective_date for row in
    bronze_customers_cdc.select("effective_date").distinct().collect()
    if row.effective_date is not None
])
print("Customer CDC days found:", cdc_days)

Customer CDC days found: ['2026-04-24', '2026-04-25', '2026-04-26', 'not_a_date']


In [0]:
for day in cdc_days:
    print(f"Processing customer CDC for day: {day}")
    day_df = bronze_customers_cdc.filter(F.col("effective_date") == day)
    apply_scd2_merge(
        cdc_day_df=day_df,
        table_name=f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2",
        business_key="customer_id",
        attr_cols=customer_attr_cols
    )

Processing customer CDC for day: 2026-04-24
  Expired changed rows + inserted 196 new active row(s) into retail_demo.silver.dim_customer_scd2
Processing customer CDC for day: 2026-04-25
  Expired changed rows + inserted 197 new active row(s) into retail_demo.silver.dim_customer_scd2
Processing customer CDC for day: 2026-04-26
  Expired changed rows + inserted 200 new active row(s) into retail_demo.silver.dim_customer_scd2
Processing customer CDC for day: not_a_date
  No valid CDC rows for retail_demo.silver.dim_customer_scd2 on this day — skipping.


## 3. Product Dimension — Initial Load + CDC

In [0]:
silver1_products = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver1_products_clean")

product_attr_cols = ["product_name", "category", "brand", "unit_price_final", "status"]

initial_scd2_load(
    clean_df=silver1_products,
    business_key="product_id",
    attr_cols=product_attr_cols,
    table_name=f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2"
)

Initial load into retail_demo.silver.dim_product_scd2: 800 rows


In [0]:
bronze_products_cdc = spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_products_cdc")

# Products CDC also needs unit_price cleaned the same way as Silver Stage 1
bronze_products_cdc_clean = (
    bronze_products_cdc
    .withColumn("unit_price_clean", F.regexp_extract(F.col("unit_price"), r"(\d+\.?\d*)", 1))
    .withColumn(
        "unit_price_final",
        F.when(F.col("unit_price_clean") == "", None).otherwise(F.col("unit_price_clean").cast("double"))
    )
)

product_cdc_days = sorted([
    row.effective_date for row in
    bronze_products_cdc_clean.select("effective_date").distinct().collect()
    if row.effective_date is not None
])
print("Product CDC days found:", product_cdc_days)

for day in product_cdc_days:
    print(f"Processing product CDC for day: {day}")
    day_df = bronze_products_cdc_clean.filter(F.col("effective_date") == day)
    apply_scd2_merge(
        cdc_day_df=day_df,
        table_name=f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2",
        business_key="product_id",
        attr_cols=product_attr_cols
    )

Product CDC days found: ['2026-04-24', '2026-04-25', '2026-04-26']
Processing product CDC for day: 2026-04-24
  Expired changed rows + inserted 60 new active row(s) into retail_demo.silver.dim_product_scd2
Processing product CDC for day: 2026-04-25
  Expired changed rows + inserted 60 new active row(s) into retail_demo.silver.dim_product_scd2
Processing product CDC for day: 2026-04-26
  Expired changed rows + inserted 60 new active row(s) into retail_demo.silver.dim_product_scd2


## 4. Verification

In [0]:
dim_customer_scd2 = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2")
dim_product_scd2 = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2")

print("dim_customer_scd2 total rows:", dim_customer_scd2.count())
print("dim_customer_scd2 current rows:", dim_customer_scd2.filter(F.col("is_current") == True).count())
print("dim_customer_scd2 distinct customer_id:", dim_customer_scd2.select("customer_id").distinct().count())
print()
print("dim_product_scd2 total rows:", dim_product_scd2.count())
print("dim_product_scd2 current rows:", dim_product_scd2.filter(F.col("is_current") == True).count())
print("dim_product_scd2 distinct product_id:", dim_product_scd2.select("product_id").distinct().count())

dim_customer_scd2 total rows: 3093
dim_customer_scd2 current rows: 2740
dim_customer_scd2 distinct customer_id: 2740

dim_product_scd2 total rows: 980
dim_product_scd2 current rows: 800
dim_product_scd2 distinct product_id: 800


### Sanity check: does any customer have MORE than one "current" row?
This should always be 0 — if it's not, the SCD2 logic has a bug.

In [0]:
dupes_check = (
    dim_customer_scd2.filter(F.col("is_current") == True)
    .groupBy("customer_id").count()
    .filter(F.col("count") > 1)
)
print("Customers with more than 1 current row (should be 0):", dupes_check.count())

Customers with more than 1 current row (should be 0): 0


### Example: view the full history of one customer who changed

In [0]:
# Pick a customer_id that appears more than once (i.e. has history)
history_example_id = (
    dim_customer_scd2.groupBy("customer_id").count()
    .filter(F.col("count") > 1)
    .select("customer_id").limit(1).collect()
)

if history_example_id:
    cid = history_example_id[0].customer_id
    print(f"History for customer_id = {cid}:")
    display(
        dim_customer_scd2.filter(F.col("customer_id") == cid)
        .select("customer_id", "customer_name", "city", "segment",
                "effective_start_date", "effective_end_date", "is_current", "surrogate_key")
        .orderBy("effective_start_date")
    )

History for customer_id = C00175:


customer_id,customer_name,city,segment,effective_start_date,effective_end_date,is_current,surrogate_key
C00175,Customer_175,Gurugram,Regular,1900-01-01,2026-04-23,false,175
C00175,Customer_175,Chennai,Platinum,2026-04-24,9999-12-31,true,2511


## Summary — Phase 3 (Silver Stage 2 — SCD Type 2)
- Built `dim_customer_scd2` and `dim_product_scd2` starting from an initial
  load of the Silver Stage 1 clean tables (every row versioned from
  1900-01-01 to 9999-12-31, marked `is_current = true`).
- Processed each day's CDC file **in chronological order**, using a
  SHA-256 hash of the tracked attributes to detect real changes (rather
  than comparing every column individually).
- For changed/new records, applied the classic **2-step MERGE pattern**:
  first expiring the old active row (`is_current = false`,
  `effective_end_date` set), then inserting a new active row with a fresh
  `surrogate_key`.
- Verified there is exactly one "current" row per business key at all
  times, and inspected a sample customer's full change history as proof
  the historical tracking works correctly.